# P3.09 Kaggle Discovery Framework

**Purpose:** Empirical capability validation for P3.09 distributed render orchestration

**Constraint:** Must complete in ≤5 minutes

**Output:** Structured JSON evidence artifact

This notebook runs discovery tests that prove what Kaggle can actually do,
independent of documentation or assumptions.

In [ ]:
import json
import os
import sys
import subprocess
import time
from datetime import datetime
from pathlib import Path

# Framework metadata
DISCOVERY_SESSION = {
    "session_id": f"discovery_{int(time.time())}",
    "timestamp": datetime.now().isoformat(),
    "timeout_hard": 300,  # 5 minutes
    "timeout_warning": 270,  # 4:30 warning
    "notebook_name": "kaggle_discovery_framework",
    "discovery_phase": "infrastructure",
    "results": []
}

start_time = time.time()

def log_test(test_name, result, evidence, duration_s):
    """Log a discovery test result."""
    DISCOVERY_SESSION["results"].append({
        "test": test_name,
        "result": result,  # PASS, FAIL, UNKNOWN, BLOCKED
        "evidence": evidence,
        "duration_s": duration_s,
        "timestamp": datetime.now().isoformat()
    })
    print(f"[{result:8s}] {test_name} ({duration_s:.1f}s)")

def check_timeout():
    """Check if we're approaching timeout."""
    elapsed = time.time() - start_time
    if elapsed > DISCOVERY_SESSION["timeout_hard"]:
        raise RuntimeError(f"HARD TIMEOUT: {elapsed:.0f}s")
    elif elapsed > DISCOVERY_SESSION["timeout_warning"]:
        print(f"⚠️  WARNING: {DISCOVERY_SESSION['timeout_hard'] - elapsed:.0f}s remaining")
    return elapsed

print(f"🔬 P3.09 Kaggle Discovery Session: {DISCOVERY_SESSION['session_id']}")
print(f"📍 Started: {DISCOVERY_SESSION['timestamp']}")

## Test 1: Environment Probe

In [ ]:
test_start = time.time()
check_timeout()

try:
    env_probe = {
        "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
        "platform": sys.platform,
        "kaggle_data_path": os.environ.get("KAGGLE_DATA_PROXY_URL", "NOT_SET"),
        "kaggle_kernel_id": os.environ.get("KAGGLE_KERNEL_ID", "NOT_SET"),
        "kaggle_working_path": Path("/kaggle/working").exists(),
        "kaggle_input_path": Path("/kaggle/input").exists(),
        "temp_writable": Path("/tmp").exists()
    }
    
    # Test file I/O
    test_file = Path("/kaggle/working/discovery_test.txt")
    test_file.write_text("discovery test")
    env_probe["file_io_working"] = test_file.read_text() == "discovery test"
    test_file.unlink()
    
    duration = time.time() - test_start
    log_test("environment_probe", "PASS", env_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("environment_probe", "FAIL", str(e), duration)

## Test 2: GPU/Accelerator Probe

In [ ]:
test_start = time.time()
check_timeout()

try:
    gpu_probe = {
        "cuda_available": False,
        "gpu_detected": False,
        "torch_available": False
    }
    
    # Try torch
    try:
        import torch
        gpu_probe["torch_available"] = True
        gpu_probe["cuda_available"] = torch.cuda.is_available()
        gpu_probe["device_count"] = torch.cuda.device_count() if gpu_probe["cuda_available"] else 0
        if gpu_probe["cuda_available"]:
            gpu_probe["device_name"] = torch.cuda.get_device_name(0)
            gpu_probe["gpu_detected"] = True
    except ImportError:
        pass
    
    # Try nvidia-smi if available
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            gpu_probe["nvidia_smi_available"] = True
    except:
        pass
    
    result_status = "PASS" if gpu_probe["gpu_detected"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("gpu_probe", result_status, gpu_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("gpu_probe", "FAIL", str(e), duration)

## Test 3: Internet/Network Probe

In [ ]:
test_start = time.time()
check_timeout()

try:
    import urllib.request
    
    network_probe = {
        "dns_working": False,
        "https_working": False,
        "http_working": False,
        "external_api_working": False
    }
    
    # Test DNS
    try:
        import socket
        socket.gethostbyname("google.com")
        network_probe["dns_working"] = True
    except:
        pass
    
    # Test HTTPS
    try:
        response = urllib.request.urlopen("https://www.google.com", timeout=5)
        network_probe["https_working"] = response.status == 200
    except:
        pass
    
    # Test external API
    try:
        response = urllib.request.urlopen("https://api.github.com", timeout=5)
        network_probe["external_api_working"] = response.status == 200
    except:
        pass
    
    result_status = "PASS" if (network_probe["dns_working"] or network_probe["https_working"]) else "BLOCKED"
    duration = time.time() - test_start
    log_test("network_probe", result_status, network_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("network_probe", "FAIL", str(e), duration)

## Test 4: Package/Runtime Probe

In [ ]:
test_start = time.time()
check_timeout()

try:
    runtime_probe = {}
    
    # Check FFmpeg
    try:
        result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True, timeout=5)
        runtime_probe["ffmpeg_available"] = result.returncode == 0
    except:
        runtime_probe["ffmpeg_available"] = False
    
    # Check ffprobe
    try:
        result = subprocess.run(["ffprobe", "-version"], capture_output=True, text=True, timeout=5)
        runtime_probe["ffprobe_available"] = result.returncode == 0
    except:
        runtime_probe["ffprobe_available"] = False
    
    # Check Node.js
    try:
        result = subprocess.run(["node", "--version"], capture_output=True, text=True, timeout=5)
        runtime_probe["node_available"] = result.returncode == 0
        if result.returncode == 0:
            runtime_probe["node_version"] = result.stdout.strip()
    except:
        runtime_probe["node_available"] = False
    
    # Check npm
    try:
        result = subprocess.run(["npm", "--version"], capture_output=True, text=True, timeout=5)
        runtime_probe["npm_available"] = result.returncode == 0
    except:
        runtime_probe["npm_available"] = False
    
    result_status = "PASS" if runtime_probe["ffmpeg_available"] else "UNKNOWN"
    duration = time.time() - test_start
    log_test("runtime_probe", result_status, runtime_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("runtime_probe", "FAIL", str(e), duration)

## Test 5: Parameter Passing / Environment Variables

In [ ]:
test_start = time.time()
check_timeout()

try:
    param_probe = {}
    
    # Check if environment variables can be read
    param_probe["shard_id_from_env"] = os.environ.get("SHARD_ID", "NOT_SET")
    param_probe["frame_start_from_env"] = os.environ.get("FRAME_START", "NOT_SET")
    param_probe["frame_end_from_env"] = os.environ.get("FRAME_END", "NOT_SET")
    
    # For now, we expect these to be NOT_SET since we haven't passed them yet
    # This is a baseline test
    # In real discovery, we'll test with actual parameter passing
    
    result_status = "UNKNOWN"  # Need to test with actual params
    duration = time.time() - test_start
    log_test("parameter_passing", result_status, param_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("parameter_passing", "FAIL", str(e), duration)

## Final Report

In [ ]:
# Calculate statistics
results = DISCOVERY_SESSION["results"]
passed = sum(1 for r in results if r["result"] == "PASS")
failed = sum(1 for r in results if r["result"] == "FAIL")
unknown = sum(1 for r in results if r["result"] == "UNKNOWN")
blocked = sum(1 for r in results if r["result"] == "BLOCKED")

total_time = time.time() - start_time

DISCOVERY_SESSION.update({
    "summary": {
        "total_tests": len(results),
        "passed": passed,
        "failed": failed,
        "unknown": unknown,
        "blocked": blocked,
        "total_time_s": total_time
    }
})

# Write artifact
output_path = Path("/kaggle/working/discovery_framework_results.json")
output_path.write_text(json.dumps(DISCOVERY_SESSION, indent=2))

print(f"\n📊 Discovery Summary:")
print(f"   PASS:    {passed}")
print(f"   FAIL:    {failed}")
print(f"   UNKNOWN: {unknown}")
print(f"   BLOCKED: {blocked}")
print(f"   Total:   {len(results)} tests in {total_time:.1f}s")
print(f"\n✅ Results saved to: {output_path}")